# LLM backtest on Kaggle GPUs

Replays `llm-backtest` with Ternary Bonsai 2 27B (PrismML llama.cpp fork) on Kaggle's T4s, then exports the cached
model answers so the backtest can be re-run at home with no model calls.

**Before running**
- Settings → Accelerator: **GPU T4 x2** (one server per T4). Internet: **On**. (Both need a phone-verified account.)
- Add Input → your **private** `stock-backtest-bundle` dataset (made by `scripts/kaggle/prepare_kaggle_bundle.py`).
- To resume a previous run: Add Input → this notebook's previous **Output**. Its `replay_cache_export.json` is
  imported before the run, so answered decisions are not asked again.
- Check setup and speed with `SMOKE_TEST_ONLY = True` and Run All. For the real run set it to `False` and use
  **Save Version → Save & Run All (Commit)**: only a committed run keeps `/kaggle/working`.
- To stop a run, press ■: the backtests stop cleanly (answers so far stay cached) and the servers keep running.

**Privacy**: keep the dataset and this notebook private. Never print secrets: cell output is saved with every version.
The database is decompressed to `/kaggle/temp`, so it is never part of the saved output; only the answer cache,
the run log and the summary are.

In [ ]:
# ---- Settings: edit these ----
MODEL_LABEL = "ternary-bonsai-2-27b-rb2048"   # use the same --model at home, or answers will not be cache hits
# Thinking: the cache key does not include these, so give each setting its own MODEL_LABEL.
REASONING_BUDGET = 2048                # hard cap on thinking tokens (~6.5 tokens/s on a T4); 0 does NOT turn it off
REASONING_EFFORT = "medium"            # "", "low", "medium" or "xhigh" ("high" errors); "medium" thinks less
# --allow-unknown-cutoff: config/model_knowledge_cutoffs.csv has no row for the local model yet, so the contamination guard
# would refuse the run. Add a row for MODEL_LABEL there (and drop the flag) to have decisions the model may remember excluded.
BACKTEST_ARGS = "--start-at 2026-09-21 --min-hours-between 4 --replay-strategist --allow-unknown-cutoff"   # the same arguments as at home
SMOKE_TEST_ONLY = True                 # True: answer the most recent decision and stop (~15-20 min; checks setup)
USE_BOTH_GPUS = True                   # one server per T4, the decisions split between them: about twice as fast
TIME_LIMIT_HOURS = 11                  # from this cell's run; stops cleanly before Kaggle's 12 h session limit
CONTEXT_TOKENS = 98304                 # strategist prompts reach ~67K tokens, plus thinking; fits a T4
PROMPT_BATCH = 2048                    # llama-server -b/-ub: larger batches read long prompts faster
PRISM_TAG = "prism-b10754-2459f68"     # the PrismML release your local server runs
CUDA_BUILDS = ["12.8", "12.4"]         # 12.8 is PrismML's recommended Linux build; 13.3 is known to segfault
HF_REPO = "prism-ml/Ternary-Bonsai-2-27B-gguf"

import time
SESSION_START = time.time()

In [ ]:
import glob, hashlib, json, os, queue, shlex, shutil, signal, subprocess, sys, tarfile, threading, urllib.request, zipfile
from pathlib import Path

TEMP, WORK = Path("/kaggle/temp"), Path("/kaggle/working")
TEMP.mkdir(exist_ok=True)
# The bundle is the folder holding manifest.json and kaggle.env. (Kaggle may have unzipped code.zip on upload.)
manifests = [Path(p) for p in glob.glob("/kaggle/input/**/manifest.json", recursive=True)
             if Path(p).with_name("kaggle.env").exists()]
if len(manifests) != 1:
    subprocess.run("find /kaggle/input -maxdepth 4 | head -40", shell=True)
assert len(manifests) == 1, f"Attach exactly one stock-backtest-bundle dataset (Input panel); found {manifests}"
BUNDLE = manifests[0].parent
MANIFEST = json.loads(manifests[0].read_text())
print("Bundle:", BUNDLE, "| commit", MANIFEST["git_commit"][:12], "| manager prompt", MANIFEST["manager_prompt_version"])
subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv"], check=True)
GPU_COUNT = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
print(f"Free disk for /kaggle/temp: {shutil.disk_usage(TEMP).free / 1e9:.0f} GB (needs ~15 GB)")

## 1. llama-server (PrismML fork)

In [ ]:
def find_server(root):
    hits = [Path(p) for p in glob.glob(f"{root}/**/llama-server", recursive=True) if os.access(p, os.X_OK)]
    return hits[0] if hits else None

def sees_cuda(server):
    env = {**os.environ, "LD_LIBRARY_PATH": f"{server.parent}:{os.environ.get('LD_LIBRARY_PATH', '')}"}
    probe = subprocess.run([str(server), "--list-devices"], capture_output=True, text=True, env=env)
    print(probe.stdout[-800:], probe.stderr[-800:])
    return probe.returncode == 0 and "CUDA" in (probe.stdout + probe.stderr)

SERVER = None
for cuda in CUDA_BUILDS:
    name = f"llama-{PRISM_TAG}-bin-linux-cuda-{cuda}-x64.tar.gz"
    root = TEMP / f"llama-cuda-{cuda}"
    if not find_server(root):
        url = f"https://github.com/PrismML-Eng/llama.cpp/releases/download/{PRISM_TAG}/{name}"
        print("Downloading", url)
        urllib.request.urlretrieve(url, TEMP / name)
        with tarfile.open(TEMP / name) as archive:
            archive.extractall(root)
    candidate = find_server(root)
    if candidate and sees_cuda(candidate):
        SERVER = candidate
        break

if SERVER is None:  # Fallback: build the same commit from source for the T4 (sm_75), ~15 min.
    print("No prebuilt binary sees the GPU; building from source")
    src = TEMP / "llama-src"
    if not src.exists():
        subprocess.run(["git", "clone", "https://github.com/PrismML-Eng/llama.cpp", str(src)], check=True)
    subprocess.run(["git", "checkout", PRISM_TAG.rsplit("-", 1)[-1]], cwd=src, check=True)
    subprocess.run(["cmake", "-B", "build", "-DGGML_CUDA=ON", "-DCMAKE_CUDA_ARCHITECTURES=75",
                    "-DGGML_CUDA_FA_ALL_QUANTS=ON", "-DLLAMA_CURL=OFF"], cwd=src, check=True)
    subprocess.run(["cmake", "--build", "build", "-j", "--target", "llama-server"], cwd=src, check=True)
    SERVER = find_server(src / "build")
    assert SERVER and sees_cuda(SERVER), "llama-server was built but does not see the GPU"
print("Using", SERVER)

## 2. Model weights (public download, checked against your local file)

In [ ]:
from huggingface_hub import hf_hub_download

MODEL = Path(hf_hub_download(HF_REPO, MANIFEST["gguf_file"], local_dir=TEMP / "models"))
digest = hashlib.sha256()
with MODEL.open("rb") as handle:
    while chunk := handle.read(1 << 24):
        digest.update(chunk)
if MANIFEST.get("gguf_sha256"):
    assert digest.hexdigest() == MANIFEST["gguf_sha256"], "Downloaded weights differ from the file you run at home"
    print("Weights match your local file:", digest.hexdigest()[:16])
else:
    print("No local hash in the manifest; downloaded", digest.hexdigest())

## 3. Start the servers (one per GPU, localhost only, no tunnel)

In [ ]:
for old in globals().get("SERVERS", []):  # re-running this cell restarts the servers
    if old.poll() is None:
        old.terminate()
        old.wait()

flags = list(MANIFEST["llama_server_flags"] or ["-ngl", "99", "-fa", "on", "-ctk", "q8_0", "-ctv", "q8_0", "--jinja"])
if "--reasoning-budget" in flags:
    flags[flags.index("--reasoning-budget") + 1] = str(REASONING_BUDGET)
else:
    flags += ["--reasoning-budget", str(REASONING_BUDGET)]
if REASONING_EFFORT:
    flags += ["--chat-template-kwargs", json.dumps({"reasoning_effort": REASONING_EFFORT})]

GPUS = list(range(min(GPU_COUNT, 2))) if USE_BOTH_GPUS else [0]
PORTS = [8080 + gpu for gpu in GPUS]
SERVERS, SERVER_LOGS = [], []
for gpu, port in zip(GPUS, PORTS):
    command = [str(SERVER), "-m", str(MODEL), *flags, "-c", str(CONTEXT_TOKENS), "-np", "1",
               "-b", str(PROMPT_BATCH), "-ub", str(PROMPT_BATCH), "--host", "127.0.0.1", "--port", str(port)]
    print(f"GPU {gpu}:", shlex.join(command))
    log_path = TEMP / f"llama-server-gpu{gpu}.log"
    env = {**os.environ, "LD_LIBRARY_PATH": f"{SERVER.parent}:{os.environ.get('LD_LIBRARY_PATH', '')}",
           "CUDA_VISIBLE_DEVICES": str(gpu)}
    # Its own session, so interrupting a cell never takes the server down with it.
    SERVERS.append(subprocess.Popen(command, stdout=log_path.open("w"), stderr=subprocess.STDOUT, env=env,
                                    start_new_session=True))
    SERVER_LOGS.append(log_path)

for gpu, port, server, log_path in zip(GPUS, PORTS, SERVERS, SERVER_LOGS):
    for _ in range(600):
        if server.poll() is not None:
            print(log_path.read_text()[-3000:])
            raise RuntimeError(f"llama-server on GPU {gpu} exited")
        try:
            if urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=2).status == 200:
                break
        except Exception:
            time.sleep(2)
    else:
        raise RuntimeError(f"llama-server on GPU {gpu} did not become healthy in 20 minutes")
    # The server's log is quiet at default verbosity, so check the GPU itself: weights + cache are several GB.
    used = int(subprocess.run(["nvidia-smi", "-i", str(gpu), "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True, check=True).stdout.strip())
    print(f"GPU {gpu}: server ready on port {port}, {used} MiB in use")
    assert used > 6000, f"The model does not appear to be on GPU {gpu}; see {log_path}"

## 4. Code, settings and database

In [ ]:
CODE = TEMP / "code"
if not CODE.exists():
    if (BUNDLE / "code.zip").is_file():
        with zipfile.ZipFile(BUNDLE / "code.zip") as archive:
            archive.extractall(CODE)
    else:  # Kaggle extracted the zip on upload: copy the checkout out of the read-only input folder.
        roots = [Path(p).parent for p in glob.glob(f"{BUNDLE}/**/requirements.txt", recursive=True)
                 if (Path(p).parent / "Python Scripts").is_dir()]
        assert len(roots) == 1, f"Could not find the code in {BUNDLE}; found {roots}"
        shutil.copytree(roots[0], CODE)
print("Code:", CODE)
shutil.copyfile(BUNDLE / "kaggle.env", CODE / ".env")   # policy settings only; no credentials

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "zstandard", "-r", str(CODE / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(CODE)], check=True)

import zstandard
DB = TEMP / "db" / "stock_experiment.db"
if not DB.exists():
    DB.parent.mkdir(exist_ok=True)
    with (BUNDLE / "stock_experiment.slim.db.zst").open("rb") as source, DB.open("wb") as target:
        zstandard.ZstdDecompressor(max_window_size=2**31).copy_stream(source, target)
print(f"Database: {DB} ({DB.stat().st_size / 1e9:.2f} GB)")

## 5. Resume from earlier runs' answers (if their output was added as input)

In [ ]:
MERGE = BUNDLE / "merge_replay_cache.py"
for export in sorted(glob.glob("/kaggle/input/**/replay_cache_export.json", recursive=True)):
    subprocess.run([sys.executable, str(MERGE), "import", export, "--db", str(DB)], check=True)

## 6. Run the backtest

With two GPUs the window is split in two at a quiet gap, chosen so the halves replay exactly the decisions one run
would (the per-symbol spacing is unchanged). Each half answers on its own GPU; when both finish, one run over the
whole window re-finalizes every decision from the cache, with no model calls, and its summary is the result.

In [ ]:
BASE_ENV = {**os.environ,
            "LLM_PROVIDER": "llamacpp", "MANAGER_PROVIDER": "llamacpp", "STRATEGIST_PROVIDER": "llamacpp",
            "MANAGER_MODEL": MODEL_LABEL, "STRATEGIST_MODEL": MODEL_LABEL,
            "DB_PATH": str(DB), "GIT_COMMIT": MANIFEST["git_commit"]}

# Runs in the project's environment: the decisions BACKTEST_ARGS selects, and a split point that keeps them.
# Note: once MODEL_LABEL has a cutoff row, the replay's guard may exclude decisions this preview counts, so the real counts can be lower.
SPLIT_WINDOW = r'''
import json, sys
from services.settings import load_process_settings
load_process_settings()
from llm_backtest import _parse_args
from market_db_modules.llm_replay_store import list_frozen_manager_decisions
from services.llm_backtest import frozen_payload, space_decisions_per_symbol
args = _parse_args()
window = list_frozen_manager_decisions(start_at=args.start_at or None, end_at=args.end_at or None, limit=args.limit)
replayable = [d for d in window if frozen_payload(d) is not None]
space = lambda ds: space_decisions_per_symbol(ds, min_hours=args.min_hours_between) if args.min_hours_between > 0 else ds
chosen = [d["id"] for d in space(replayable)]
best = None
if args.limit is None and len(chosen) >= 4:
    for i in range(1, len(replayable)):
        first, second = replayable[:i], replayable[i:]
        if first[-1]["decision_run_at"] == second[0]["decision_run_at"]:
            continue
        a, b = space(first), space(second)
        if [d["id"] for d in a + b] == chosen and (best is None or abs(len(a) - len(b)) < best[0]):
            best = (abs(len(a) - len(b)), first[-1]["decision_run_at"], second[0]["decision_run_at"], len(a), len(b))
print(json.dumps({"decisions": len(chosen), "split": best and {"end_at": best[1], "start_at": best[2], "counts": best[3:]}}))
'''

def split_window():
    out = subprocess.run([sys.executable, "-c", SPLIT_WINDOW, *shlex.split(BACKTEST_ARGS)], cwd=CODE, env=BASE_ENV,
                         capture_output=True, text=True)
    lines = [line for line in out.stdout.splitlines() if line.startswith("{")]
    if out.returncode != 0 or not lines:
        print(out.stdout[-2000:], out.stderr[-2000:])
        raise RuntimeError("Could not list the decisions in BACKTEST_ARGS")
    return json.loads(lines[-1])

def start(extra_args, hours, port):
    # The entrypoint itself in its own session: the time limit's SIGINT reaches the backtest, and ■ does not.
    command = ["timeout", "--signal=INT", f"{max(hours, 0.1):.2f}h", sys.executable,
               "Python Scripts/agentCallers/llm_backtest.py", "--model", MODEL_LABEL, *shlex.split(extra_args)]
    print(f"[port {port}]", shlex.join(command))
    return subprocess.Popen(command, cwd=CODE, env={**BASE_ENV, "LLAMACPP_BASE_URL": f"http://127.0.0.1:{port}/v1"},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, start_new_session=True)

def heartbeat():
    # What each server is doing now: a decision prints nothing until both its model calls finish (~15-20 min).
    for gpu, path in zip(GPUS, SERVER_LOGS):
        timing = [line for line in path.read_text(errors="replace").splitlines()[-40:] if "print_timing" in line]
        status = timing[-1].split("|", 2)[-1].strip() if timing else "idle"
        print(f"  ... {time.strftime('%H:%M')} GPU {gpu}: {status}")

def follow(processes):
    """Stream each process's output (tagged when several run) to the cell and run.log; return each final JSON summary."""
    lines, done = queue.Queue(), {name: [] for name in processes}
    def pump(name, process):
        for line in process.stdout:
            lines.put((name, line))
        lines.put((name, None))
    for item in processes.items():
        threading.Thread(target=pump, args=item, daemon=True).start()
    running = len(processes)
    try:
        with (WORK / "run.log").open("a") as log:
            while running:
                try:
                    name, line = lines.get(timeout=120)
                except queue.Empty:
                    heartbeat()
                    continue
                if line is None:
                    running -= 1
                    continue
                tagged = f"[{name}] {line}" if len(processes) > 1 else line
                print(tagged, end="")
                log.write(tagged)
                log.flush()
                done[name].append(line)
    except KeyboardInterrupt:  # stop the backtests cleanly; answers so far are already cached
        for process in processes.values():
            if process.poll() is None:
                process.send_signal(signal.SIGINT)
        raise
    finally:
        for process in processes.values():
            process.wait()
    return {name: summary_of("".join(output)) for name, output in done.items()}

def summary_of(text):
    start_at = text.rfind("\n{\n")
    try:
        return json.loads(text[start_at + 1:]) if start_at >= 0 else None
    except json.JSONDecodeError:
        return None

def hours_left():
    return TIME_LIMIT_HOURS - (time.time() - SESSION_START) / 3600

def export_answers():
    subprocess.run([sys.executable, str(MERGE), "export", "--db", str(DB), "--out", str(WORK / "replay_cache_export.json"),
                    "--manifest", str(BUNDLE / "manifest.json")], check=True)

try:
    if SMOKE_TEST_ONLY:
        summary = follow({"smoke": start(f"{BACKTEST_ARGS} --limit 1", min(1, hours_left()), PORTS[0])})["smoke"]
        print("Smoke test:", json.dumps(summary, indent=2) if summary else "no summary (see log above)")
    else:
        plan = split_window()
        split = plan["split"] if len(PORTS) > 1 else None
        print(f"{plan['decisions']} decisions to replay;",
              f"split {split['counts'][0]} + {split['counts'][1]} across two GPUs" if split else "one GPU")
        if split:
            halves = follow({"gpu0": start(f"{BACKTEST_ARGS} --end-at {split['end_at']}", hours_left(), PORTS[0]),
                             "gpu1": start(f"{BACKTEST_ARGS} --start-at {split['start_at']}", hours_left(), PORTS[1])})
            # Both halves done: every answer is cached, so this pass makes no model calls (barring a failed answer).
            summary = (follow({"all": start(BACKTEST_ARGS, max(hours_left(), 0.5), PORTS[0])})["all"]
                       if all(halves.values()) else None)
        else:
            summary = follow({"run": start(BACKTEST_ARGS, hours_left(), PORTS[0])})["run"]
        (WORK / "summary.json").write_text(json.dumps(summary, indent=2) if summary else '{"stopped": true}')
        print("Finished" if summary else "Stopped at the time limit; commit again with this output added to resume")
finally:
    export_answers()   # every cached answer, even after an error or the time limit

## 7. Back home

1. Download `replay_cache_export.json` from this version's **Output** tab.
2. `python3 scripts/kaggle/merge_replay_cache.py import replay_cache_export.json --db <the DB you backtest against>`
3. Re-run the same command with `LLM_PROVIDER=llamacpp` and the same `--model` (MODEL_LABEL): every answer is now a
   cache hit (`cached_answer_count` equals the decision count), so no GPU is needed. Add `--fetch-option-quotes`
   there, so your MarketData token never leaves your machine.

## Clean-up: stops any backtest still running (e.g. from an older version of this notebook) and the servers

In [ ]:
subprocess.run(["pkill", "-INT", "-f", "agentCallers/llm_backtest.py"])

for server in SERVERS:
    server.terminate()